# Clinical Analytics Chatbot
**Drug R&D Assistant — Dataiku Notebook UI**

Run all cells in order. The interactive chat UI launches in the last cell (Cell 4).

**Before running:** set your LLM Mesh connection ID in Cell 2.

`lib/python/` is automatically on the Python path in all Dataiku notebooks — no path manipulation needed.

In [ ]:
# Cell 1 — Initialise Panel
import panel as pn
pn.extension('bokeh', sizing_mode='stretch_width', raw_css=["""
[class*="chat"] {
    max-width: 100% !important;
    width: 100% !important;
    box-sizing: border-box !important;
}
[class*="chat"] > div, [class*="chat"] > div > div {
    max-width: 100% !important;
    width: 100% !important;
}
"""])

In [ ]:
# Cell 2 — Configuration
# Replace with your actual Dataiku LLM Mesh connection ID
LLM_CONNECTION_ID = "YOUR_LLM_CONNECTION_ID"

In [ ]:
# Cell 3 — Initialise backend
import yaml
from pathlib import Path
import backend

config_dir = Path(backend.__file__).parent.parent / "config"
with open(config_dir / "llm_config.yaml") as f:
    config = yaml.safe_load(f)
config["llm_mesh"]["connection_id"] = LLM_CONNECTION_ID

from backend.state.session_store import SessionStore
from backend.orchestrator.orchestrator import Orchestrator

session_store = SessionStore(timeout_minutes=60)
orchestrator = Orchestrator(session_store, config=config)

print(f"Skills loaded: {list(orchestrator.schemas.keys())}")
print(f"LLM connection: {orchestrator.llm.connection_id}")

try:
    _test = orchestrator.llm.complete([{"role": "user", "content": "Reply with the single word: OK"}])
    print(f"LLM test PASSED: {_test[:80]!r}")
except Exception as _e:
    print(f"LLM test FAILED: {_e}")

In [ ]:
# Cell 4 — Build and launch the interactive UI
import uuid
import pandas as pd
import panel as pn
from panel.chat import ChatInterface
from backend.state.conversation_state import FSMState
from backend.agents.site_list_merger_agent import parse_uploaded_file

SESSION_ID = str(uuid.uuid4())


# ── LLM trace log ────────────────────────────────────────────────────────────
_trace_content = pn.pane.Markdown(
    '*No LLM calls yet — send a message to see the trace.*',
    sizing_mode='stretch_width',
    styles={'font-size': '11px', 'white-space': 'pre-wrap', 'font-family': 'monospace', 'color': '#222'},
)


def _infer_call_label(messages):
    system = next((m["content"] for m in messages if m["role"] == "system"), "").lower()
    if system.startswith("[citeline semantic mapping]"):  return "Citeline Semantic Mapping"
    if system.startswith("[citeline filter result]"):     return "Citeline Filter Result"
    if "routing assistant" in system or ("intent" in system and "skill" in system):
        return "Intent Classification"
    if "parameter extraction" in system: return "Parameter Extraction"
    if "site list" in system or "ctms" in system: return "Site List Matching Agent"
    if "benchmarking" in system or "benchmark" in system: return "Trial Benchmarking Agent"
    if "reimbursement" in system or "hta" in system: return "Drug Reimbursement Agent"
    if "enrollment" in system and ("pessimistic" in system or "scenario" in system):
        return "Enrollment Params Estimation"
    if "narrative" in system or ("enrollment" in system and "interpret" in system):
        return "Enrollment Narrative"
    if system.startswith("["):  return system.strip("[]").title()
    return "LLM Call"


def _update_trace_log():
    log = getattr(orchestrator.llm, "call_log", [])
    conn_id = getattr(orchestrator.llm, "connection_id", "unknown")
    if not log:
        _trace_content.object = (
            f"Connection ID in use: {conn_id}\n\n"
            "No LLM calls recorded yet."
        )
        return
    entries = []
    for i, entry in enumerate(log, 1):
        msgs      = entry.get("messages", [])
        resp_text = entry.get("response", "")
        is_error  = entry.get("error", False)
        is_synth  = entry.get("synthetic", False)
        label     = _infer_call_label(msgs)
        if is_synth:
            lines = [f"=== {i}: {label} ===", resp_text]
        elif is_error:
            lines = [f"!!! ERROR Call {i}: {label} (conn={conn_id}) !!!"]
            for msg in msgs:
                lines.append(f"[{msg.get('role','').upper()}]\n{msg.get('content','')[:600]}")
            lines.append(f"[RESPONSE]\n{resp_text[:800]}")
        else:
            lines = [f"--- Call {i}: {label} ---"]
            for msg in msgs:
                content = msg.get('content', '')
                lines.append(f"[{msg.get('role','').upper()}]\n{content[:600]}"
                             + ("...[truncated]" if len(content) > 600 else ""))
            lines.append(f"[RESPONSE]\n{resp_text[:800]}"
                        + ("...[truncated]" if len(resp_text) > 800 else ""))
        entries.append("\n".join(lines))
    _trace_content.object = "\n\n".join(entries)


# ── Response → Panel ─────────────────────────────────────────────────────────
def _response_to_panel(resp):
    items = []

    if resp.get('message'):
        items.append(pn.pane.Markdown(resp['message'], sizing_mode='stretch_width'))

    if resp.get('table_data'):
        rows    = resp['table_data']
        columns = resp.get('table_columns')
        df = pd.DataFrame(rows, columns=columns) if columns else pd.DataFrame(rows)
        items.append(pn.pane.DataFrame(
            df, sizing_mode='stretch_width', max_rows=200,
            styles={'font-size': '12px'},
        ))

    if resp.get('chart_json'):
        _chart = resp['chart_json']
        if isinstance(_chart, dict):
            items.append(pn.pane.Markdown('*Chart unavailable in notebook — use the webapp.*'))
        else:
            items.append(pn.pane.Bokeh(_chart, sizing_mode='stretch_width'))

    if resp.get('fsm_state') == 'confirmation_pending':
        yes_btn = pn.widgets.Button(name='\u2713  Yes, proceed', button_type='success',
                                    width=160, margin=(8, 6, 4, 0))
        no_btn  = pn.widgets.Button(name='\u2717  Cancel', button_type='danger',
                                    width=110, margin=(8, 0, 4, 0))

        def _confirm(event):
            yes_btn.disabled = no_btn.disabled = True
            r = orchestrator.handle_confirmation(SESSION_ID, confirmed=True)
            chat.send(_response_to_panel(r), user='Assistant', respond=False)
            _maybe_show_export(r)

        def _cancel(event):
            yes_btn.disabled = no_btn.disabled = True
            r = orchestrator.handle_confirmation(SESSION_ID, confirmed=False)
            chat.send(_response_to_panel(r), user='Assistant', respond=False)

        yes_btn.on_click(_confirm)
        no_btn.on_click(_cancel)
        items.append(pn.Row(yes_btn, no_btn, margin=(4, 0, 0, 0)))

    if resp.get('result_id'):
        _maybe_show_export(resp)

    return pn.Column(*items, sizing_mode='stretch_width') if items else pn.pane.Markdown('...')


# ── Export bar ───────────────────────────────────────────────────────────────
_current_result_id = None
export_input  = pn.widgets.TextInput(placeholder='Dataiku dataset name\u2026', width=260)
export_button = pn.widgets.Button(name='\u2b07  Export to Dataiku Dataset',
                                   button_type='primary', width=220)
export_status = pn.pane.Markdown('', width=400)
export_row = pn.Column(
    pn.pane.Markdown('**Export last result:**', margin=(0, 0, 4, 0)),
    pn.Row(export_input, export_button, export_status),
    visible=False,
    styles={'background': '#f0f4ff', 'padding': '8px', 'border-radius': '6px'},
)

def _maybe_show_export(resp):
    global _current_result_id
    if resp.get('result_id'):
        _current_result_id = resp['result_id']
        export_row.visible = True

def _on_export(event):
    if not export_input.value.strip():
        export_status.object = '\u26a0 Enter a dataset name first.'
        return
    if not _current_result_id:
        export_status.object = '\u26a0 No result to export.'
        return
    r = orchestrator.export_to_dataset(SESSION_ID, _current_result_id, export_input.value.strip())
    export_status.object = r.get('message', 'Done.')

export_button.on_click(_on_export)


# ── Site list file upload (for Site List Matching) ───────────────────────────
upload_widget = pn.widgets.FileInput(accept='.csv,.xlsx,.xls', width=260)
upload_status = pn.pane.Markdown('', sizing_mode='stretch_width',
                                  styles={'font-size': '12px', 'color': '#444'})

class _FakeFileStorage:
    def __init__(self, filename, data):
        self.filename = filename
        self._data = data
    def read(self):
        return self._data

def _on_upload(event):
    if upload_widget.value is None:
        return
    filename = upload_widget.filename or 'upload'
    try:
        parsed = parse_uploaded_file(_FakeFileStorage(filename, upload_widget.value))
        state = orchestrator.session_store.get_or_create(SESSION_ID)
        state.uploaded_files['site_file'] = parsed
        state.active_skill = 'site_list_matching'
        state.fsm_state = FSMState.PARAMETER_GATHERING
        upload_status.object = (
            f'\u2705 **{filename}** — {len(parsed["data"])} rows, '
            f'columns: {", ".join(parsed["columns"])}'
        )
        chat.send(
            f'Site list uploaded: **{filename}** ({len(parsed["data"])} rows). '
            f'Type \"match\" or \"proceed\" to run matching against the CTMS database.',
            user='Assistant', respond=False,
        )
    except ValueError as e:
        upload_status.object = f'\u26a0 **Upload error:** {e}'

upload_widget.param.watch(_on_upload, 'value')

upload_bar = pn.Row(
    pn.pane.Markdown('\U0001f4c2 **Site List** *(for Site List Matching)*',
                     margin=(6, 10, 0, 0),
                     styles={'font-size': '13px', 'white-space': 'nowrap'}),
    upload_widget,
    upload_status,
    sizing_mode='stretch_width',
    styles={'background': '#f9f9f9', 'padding': '8px 12px',
            'border-bottom': '1px solid #e0e0e0'},
    align='center',
)


# ── Chat callback ─────────────────────────────────────────────────────────────
def chat_callback(contents, user, instance):
    try:
        resp = orchestrator.process_message(SESSION_ID, contents)
    except Exception as _exc:
        import traceback
        _trace_content.object = f"EXCEPTION:\n{traceback.format_exc()}"
        return pn.pane.Markdown(f"**Error:** `{_exc}`  \nSee trace pane for details.")
    _maybe_show_export(resp)
    _update_trace_log()
    return _response_to_panel(resp)


chat = ChatInterface(
    callback=chat_callback,
    user='You',
    avatar='\U0001f464',
    callback_user='Assistant',
    show_rerun=False,
    show_undo=False,
    show_copy_icon=False,
    placeholder_text='Describe what you need\u2026',
    sizing_mode='stretch_width',
    height=720,
    styles={'border': '1px solid #e0e0e0', 'border-radius': '8px'},
)

chat.send(
    pn.pane.Markdown(
        "Hello! I'm your **Clinical Analytics Assistant**. I can help you with:\n\n"
        "1. **Site List Matching** \u2014 Upload a site list and match it against the CTMS master database  \n"
        "2. **Trial Benchmarking** \u2014 Benchmark trials by indication, age group, and phase  \n"
        "3. **Drug Reimbursement** \u2014 Assess reimbursement outlook by country  \n"
        "4. **Enrollment Forecasting** \u2014 Generate pessimistic / moderate / optimistic enrollment curves  \n\n"
        "What would you like to do?"
    ),
    user='Assistant', respond=False,
)


# ── App layout ────────────────────────────────────────────────────────────────
app = pn.Column(
    pn.pane.Markdown(
        '# Clinical Analytics Assistant\n*Drug R&D | Dataiku Notebook*',
        styles={'background': '#1a1a2e', 'color': 'white',
                'padding': '14px 18px', 'border-radius': '10px 10px 0 0',
                'margin': '0'},
    ),
    upload_bar,
    chat,
    export_row,
    pn.Column(
        pn.pane.Markdown('### LLM Call Trace', margin=(0, 0, 4, 0),
                         styles={'color': '#222'}),
        _trace_content,
        sizing_mode='stretch_width',
        height=320,
        scroll=True,
        styles={'background': '#f7f7f7', 'padding': '10px',
                'border': '1px solid #ddd', 'border-radius': '6px'},
    ),
    sizing_mode='stretch_width',
    styles={'border': '1px solid #ccc', 'border-radius': '10px', 'overflow': 'hidden'},
)

app.servable()
app